<a href="https://colab.research.google.com/github/webpug/geospatial/blob/main/portal_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌀 The Portal Lab

*Pick any point on Earth. Step through, and see where you come out.*

The last notebook established one fact: every 10 × 10 m square of the planet carries a
64-number fingerprint, and two places with similar fingerprints look, function, and feel
similar — no labels required. This notebook asks the only question worth asking next:

> **"Where else is exactly like here?"**

Three experiments, each one a step closer to a toy app:

1. **Continental twins** — your point's most similar place on every continent. Including Antarctica.
2. **The bounded hunt** — draw any region on a map, find the most *here-like* spot inside it.
3. **Ghost walks** — draw a walk, and find the top-N places on Earth where that same walk exists.

Everything is self-contained — you do not need to run the other notebook first.

---

In [ ]:
#@title ▶️ Run me: install and connect { display-mode: "form" }
!pip install -q earthengine-api geemap

# Colab needs this one line or the interactive maps show up blank.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("✅ Colab interactive maps enabled")
except Exception:
    pass

# Use your own Cloud project id here (any project with Earth Engine enabled).
PROJECT_ID = "earthai-503820"  #@param {type:"string"}

import ee
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print("✅ Connected. You now have the planet on tap.")

In [ ]:
#@title ▶️ Run me: make the charts pretty { display-mode: "form" }
import numpy as np, pandas as pd, matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

NAVY, AMBER, CRIMSON, PAPER = "#0A1F3D", "#E8A33D", "#C83232", "#F8F8F5"
HEAT = LinearSegmentedColormap.from_list("heat", [NAVY, "#3d5a80", AMBER, CRIMSON])

mpl.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "savefig.dpi": 160, "axes.edgecolor": "#c9c8bf", "axes.linewidth": 1.0,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 15, "axes.titleweight": "bold", "axes.titlecolor": NAVY,
    "axes.labelsize": 11.5, "axes.labelcolor": "#4b4a46",
    "xtick.color": "#6b6a64", "ytick.color": "#6b6a64",
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "grid.color": "#dedcd3", "grid.linewidth": 0.8,
    "legend.frameon": False, "font.size": 11,
    "figure.constrained_layout.use": True,
})

def caption(ax, text):
    ax.annotate(text, xy=(0, -0.16), xycoords="axes fraction",
                fontsize=9.5, color="#8b8981", va="top", wrap=True)

# One map helper, so maps work whether or not Colab widgets cooperate.
def new_map(center, zoom, height="560px"):
    try:
        import geemap
        return geemap.Map(center=center, zoom=zoom, height=height)
    except Exception:
        import geemap.foliumap as gfm
        print("(using the static-friendly map, still pan and zoom)")
        return gfm.Map(center=center, zoom=zoom, height=height)

print("✅ Charts will come out looking good.")

In [ ]:
#@title ▶️ Run me: the search engine { display-mode: "form" }
import io, math, urllib.request

BANDS = [f"A{i:02d}" for i in range(64)]

def embedding(year):
    return (ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
              .filterDate(f"{year}-01-01", f"{year+1}-01-01")
              .mosaic().select(BANDS))

def fingerprint_at(img, lat, lon):
    # the 64 numbers for one 10 m square, or None if there is no data there
    raw = img.reduceRegion(ee.Reducer.first(), ee.Geometry.Point([lon, lat]), 10).getInfo()
    if raw.get("A00") is None:
        return None
    return np.array([raw[b] for b in BANDS], float)

def sim_image(img, vec):
    # dot product of one fingerprint against every pixel on Earth, as an image
    return (img.multiply(ee.Image.constant([float(v) for v in vec]))
               .reduce(ee.Reducer.sum()).rename("sim"))

def best_match(sim, geom, scales=(20000, 2500, 300, 10), exclude=None):
    # Coarse-to-fine argmax. Earth Engine has no "argmax" reducer, but
    # max over the bands [sim, lon, lat] returns the max sim AND its coordinates.
    img = sim.addBands(ee.Image.pixelLonLat())
    if exclude is not None:
        img = img.updateMask(ee.Image.constant(1).clip(exclude).mask().Not())
    area, hit = geom, None
    for s in scales:
        r = img.reduceRegion(ee.Reducer.max(3), area, s,
                             maxPixels=int(1e13), tileScale=4).getInfo()
        if r.get("max") is None:
            return hit                       # no data at this zoom; keep the last good level
        hit  = {"score": r["max"], "lon": r["max1"], "lat": r["max2"], "found_at_scale": s}
        area = ee.Geometry.Point([hit["lon"], hit["lat"]]).buffer(s * 3).bounds()
    return hit

def scale_ladder(geom):
    # pick a sensible coarse-to-fine ladder for a region of any size
    side_m = math.sqrt(geom.area(1000).getInfo())
    s, ladder = min(max(side_m / 250, 10), 25000), []
    while s > 10:
        ladder.append(int(round(s)))
        s /= 8
    ladder.append(10)
    return ladder

def sample_image(img, pts, scale):
    # sample any image at an array of [lon, lat] points, in order, batched
    feats = [ee.Feature(ee.Geometry.Point([float(x), float(y)]), {"_i": i})
             for i, (x, y) in enumerate(pts)]
    props = [None] * len(feats)
    for k in range(0, len(feats), 3500):
        fc = img.reduceRegions(ee.FeatureCollection(feats[k:k + 3500]),
                               ee.Reducer.first(), scale, tileScale=4)
        for f in fc.getInfo()["features"]:
            props[f["properties"]["_i"]] = f["properties"]
    return props

def drawn_geometry(m, kind):
    # whatever the user drew on a geemap map, if anything
    g = getattr(m, "user_roi", None)
    if g is None:
        return None
    try:
        info = g.getInfo()
    except Exception:
        return None
    return info["coordinates"] if info.get("type") == kind else None

def year_photo(year):
    return (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterDate(f"{year}-01-01", f"{year+1}-01-01")
              .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
              .median().select(["B4", "B3", "B2"]))

def photo(rgb, lat, lon, width_m=2200):
    box = ee.Geometry.Point([lon, lat]).buffer(width_m / 2).bounds()
    url = rgb.getThumbURL({"region": box, "dimensions": 256, "format": "png",
                           "min": 0, "max": 3000})
    return plt.imread(io.BytesIO(urllib.request.urlopen(url).read()), format="png")

def photo_strip(items, cols=4, width_m=2200):
    # items: list of (title, lat, lon)
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.1 * cols, 3.4 * rows))
    for ax in np.ravel(axes):
        ax.axis("off")
    for ax, (title, lat, lon) in zip(np.ravel(axes), items):
        try:
            ax.imshow(photo(RGB, lat, lon, width_m))
        except Exception:
            ax.text(0.5, 0.5, "no photo\n(clouds or no coverage)", ha="center",
                    va="center", transform=ax.transAxes, color="#8b8981", fontsize=9)
        ax.set_title(title, fontsize=10.5, color=NAVY)
    plt.show()

def maps_link(lat, lon):
    return f"https://maps.google.com/?q={lat:.5f},{lon:.5f}"

print("✅ Search engine loaded.")

> ### 💡 How the search actually works
>
> "Find the most similar pixel on Earth" is an argmax over roughly a **trillion**
> candidates. Nobody scans a trillion pixels interactively. The trick is a zoom ladder:
>
> **20 km → 2.5 km → 300 m → 10 m**
>
> At 20 km, Earth Engine hands us *averaged* fingerprints (the image pyramid), so one pass
> over a whole continent is cheap. We take the winner, cut a small window around it, and
> re-run at the next finer scale — binoculars first, magnifier last.
>
> **The honest caveat:** coarse levels are neighbourhood averages. A lone perfect 10 m twin
> buried inside a very *different* neighbourhood gets blurred away and can escape the net.
> That gap between "fast" and "exhaustive" is a real research problem — approximate
> nearest-neighbour search over a planet — and the roadmap at the bottom comes back to it.

In [ ]:
#@title ▶️ Run me: choose your portal { display-mode: "form" }
PORTAL_NAME = "Chicago lakefront"  #@param {type:"string"}
PORTAL_LAT  = 41.8827   #@param {type:"number"}
PORTAL_LON  = -87.6233  #@param {type:"number"}
YEAR        = 2024      #@param {type:"integer"}
EXCLUDE_KM  = 150       #@param {type:"integer"}

emb  = embedding(YEAR)
RGB  = year_photo(YEAR)
qvec = fingerprint_at(emb, PORTAL_LAT, PORTAL_LON)
assert qvec is not None, "No fingerprint here — probably open water. Nudge the coordinates."

sim         = sim_image(emb, qvec)
home_bubble = ee.Geometry.Point([PORTAL_LON, PORTAL_LAT]).buffer(EXCLUDE_KM * 1000)

print(f"🌀 Portal open at {PORTAL_NAME}  ({PORTAL_LAT:.4f}, {PORTAL_LON:.4f}), year {YEAR}.")
print(f"   Fingerprint length {np.linalg.norm(qvec):.3f}.")
print(f"   Anything within {EXCLUDE_KM} km of home is disqualified — twins must be elsewhere.")

---

## Part 1 — 🗺️ Continental twins

One search per continent, Antarctica included. Each runs the full zoom ladder, so give it
a few minutes. The continent outlines below are hand-drawn and deliberately rough — ocean
slop is free (water has no fingerprints), and only the land borders matter. A few border
regions (the Caucasus, Indonesia/New Guinea, Central America, scattered Pacific islands)
are approximate; every result comes with coordinates and a map link so you can judge.

In [ ]:
#@title ▶️ Run me: search all seven continents (takes a few minutes) { display-mode: "form" }
FAST_MODE = False  #@param {type:"boolean"}

P = lambda ring: ee.Geometry.Polygon([ring], None, False)
CONTINENTS = {
    "Africa": P([[-26,36.5],[10,38.5],[12,34],[32.4,31.5],[34.8,28],[43.5,12.5],
                 [52,12],[60,-27],[35,-38],[10,-38],[-26,0]]),
    "Europe": P([[-31,34],[-5.3,35.9],[10.5,37.9],[12.2,35],[21,35.5],[26.5,34],
                 [26.2,40],[29.2,41.1],[36.6,45],[49.5,44.5],[52,47],[60,52],
                 [60,69],[69,77],[69,82],[-31,82]]),
    "Asia": P([[26.5,34],[32.4,31.5],[34.8,28],[43.5,12.5],[51,11],[60,6],[80,4],
               [95,-8],[105,-12],[130,-12],[130,2],[179.9,2],[179.9,82],[69,82],
               [69,77],[60,69],[60,52],[52,47],[49.5,44.5],[36.6,45],[29.2,41.1],[26.2,40]]),
    "North America": P([[-168,52],[-135,30],[-115,15],[-92,12],[-83,6.8],[-78,7.0],
                        [-77.2,8.8],[-71,13],[-61.6,11.3],[-55,25],[-30,59],[-30,67],
                        [-25,72],[-11,83.5],[-168,84]]),
    "South America": P([[-95,2],[-83,6.8],[-78,7.0],[-77.2,8.8],[-71,13],[-61.6,11.3],
                        [-52,6],[-25,0],[-25,-60],[-95,-60]]),
    "Oceania": ee.Geometry.MultiPolygon(
        [[[[110,-50],[110,-11.5],[130,-11.5],[130,2],[179.9,2],[179.9,-50]]],
         [[[-179.9,-50],[-179.9,25],[-128,25],[-128,-50]]]], None, False),
    "Antarctica": ee.Geometry.Rectangle([-179.9,-89.9,179.9,-60], None, False),
}

SCALES = (20000, 2500, 300) if FAST_MODE else (20000, 2500, 300, 10)

results = []
for name, geom in CONTINENTS.items():
    print(f"   🔎 {name:15s}", end=" ")
    hit = best_match(sim, geom, scales=SCALES, exclude=home_bubble)
    if hit is None:
        print("… no fingerprints here at all")
        continue
    print(f"… twin found, similarity {hit['score']:+.3f}")
    results.append({"continent": name, **hit})

R = pd.DataFrame(results).sort_values("score", ascending=False).reset_index(drop=True)
R["view"] = [maps_link(a, b) for a, b in zip(R.lat, R.lon)]

print("\n" + "=" * 78)
print(f"  THE SEVEN DOORS OUT OF {PORTAL_NAME.upper()}")
print("=" * 78)
print(R[["continent", "score", "lat", "lon", "view"]].to_string(index=False))
print("=" * 78)
print("  1.000 would be a perfect twin. Click the links. Trust nothing, check everything.")

In [ ]:
#@title ▶️ Run me: the doors, on a map { display-mode: "form" }
m_world = new_map([20, 0], 2)
m_world.add_basemap("CartoDB.DarkMatter")
twins = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([r.lon, r.lat]), {"continent": r.continent})
    for r in R.itertuples()])
m_world.addLayer(twins.style(color="E8A33D", pointSize=7), {}, "the twins")
m_world.addLayer(ee.Geometry.Point([PORTAL_LON, PORTAL_LAT]), {"color": "white"}, "your portal")
m_world

In [ ]:
#@title ▶️ Run me: what the doors look like { display-mode: "form" }
items = [(f"YOUR PORTAL\n{PORTAL_NAME}", PORTAL_LAT, PORTAL_LON)]
items += [(f"{r.continent}\nsim {r.score:+.3f}", r.lat, r.lon) for r in R.itertuples()]
photo_strip(items, cols=4)

### 👀 What to look for

- **The ranking is a climate story.** A Chicago portal usually finds its best doors in
  Europe and Asia — same latitude band, same seasons, same kind of city. The Africa and
  Oceania doors land in the temperate corners (South Africa, southeast Australia).
- **Antarctica is the control.** Unless your portal is ice, rock, or tundra, its score
  should be dramatically lower. If it is not, be suspicious and look at the photo.
- **The photos are the test.** The model never saw "city" or "farm" as words. If the
  photo strip still *rhymes*, the fingerprints are carrying real meaning.
- A twin found in `FAST_MODE` stops at 300 m — good enough to explore, not pixel-exact.

---

## Part 2 — ✏️ The bounded hunt

The planet is a big haystack. Sometimes the question is smaller: *"where is the most
here-like spot in Japan?"* or *"…inside this valley?"* Draw the region yourself.

In [ ]:
#@title ▶️ Run me: draw your hunting ground { display-mode: "form" }
# Use the rectangle or polygon tool in the map toolbar, draw ONE region, any size.
# If drawing tools do not appear (static map fallback), skip ahead - the next cell
# has a typed fallback box.
m_region = new_map([30, 0], 2)
m_region.add_basemap("Esri.WorldImagery")
print("✏️  Draw one rectangle or polygon, then run the next cell.")
m_region

In [ ]:
#@title ▶️ Run me: hunt inside it { display-mode: "form" }
FALLBACK_BOX = "129, 30, 146, 45.8"  #@param {type:"string"}
# fallback is [west, south, east, north] - the default is Japan

coords = drawn_geometry(m_region, "Polygon") if "m_region" in dir() else None
if coords:
    roi, source = ee.Geometry.Polygon(coords, None, False), "your drawing"
else:
    w, s, e, n = [float(x) for x in FALLBACK_BOX.split(",")]
    roi, source = ee.Geometry.Rectangle([w, s, e, n], None, False), f"fallback box {FALLBACK_BOX}"

ladder = scale_ladder(roi)
print(f"Hunting inside {source}  ·  zoom ladder {ladder}")

hit = best_match(sim, roi, scales=ladder, exclude=home_bubble)
assert hit is not None, "No fingerprints inside that region (all water?). Draw another."

print(f"\n🎯 Best match: ({hit['lat']:.5f}, {hit['lon']:.5f})   similarity {hit['score']:+.3f}")
print(f"   {maps_link(hit['lat'], hit['lon'])}")

photo_strip([(f"YOUR PORTAL\n{PORTAL_NAME}", PORTAL_LAT, PORTAL_LON),
             (f"best match in region\nsim {hit['score']:+.3f}", hit["lat"], hit["lon"])], cols=2)

m_hit = new_map([hit["lat"], hit["lon"]], 12)
m_hit.add_basemap("Esri.WorldImagery")
box = ee.Geometry.Point([hit["lon"], hit["lat"]]).buffer(20000).bounds()
m_hit.addLayer(sim.clip(box),
               {"min": 0.3, "max": 1.0, "palette": ["0A1F3D", "3d5a80", "E8A33D", "C83232"],
                "opacity": 0.55},
               "similarity heat")
m_hit.addLayer(ee.Geometry.Point([hit["lon"], hit["lat"]]), {"color": "white"}, "the match")
m_hit

### 👀 What to look for

The heat overlay around the winner shows the *runner-up structure* — often the match is
not a lone pixel but a whole district of near-twins. That structure is the difference
between "this exact spot" and "this kind of place", and it is what makes Part 3 possible.

---

## Part 3 — 👣 Ghost walks

A place is one fingerprint. A **walk** is a *sequence* of fingerprints — shore, then park,
then towers, in that order, at that spacing. So: draw a line, and hunt the planet for
somewhere you could take the *same walk*.

The recipe, honestly stated:

1. Resample your line into evenly spaced steps and fingerprint each step.
2. **Blur:** scan the whole planet at 25 km for regions matching the *average* fingerprint
   of the walk — the candidate shortlist.
3. **Try the walk everywhere plausible:** at each candidate, slide the exact shape of your
   line around a grid of positions and rotations, scoring each placement step-by-step
   against smoothed 240 m fingerprints.
4. **Focus:** jitter the survivors at 60 m, then score the finalists at the true 10 m.

Same shape, same length, any position, any rotation. No mirroring, no stretching — those
are v1 features. This is the most research-grade part of the notebook: expect it to find
the right *kind* of place reliably, and the exact best path only sometimes.

In [ ]:
#@title ▶️ Run me: draw your walk { display-mode: "form" }
# Use the polyline tool. A walk of 2-30 km works best. If drawing is unavailable,
# the next cell falls back to typed coordinates.
m_walk = new_map([PORTAL_LAT, PORTAL_LON], 12)
m_walk.add_basemap("Esri.WorldImagery")
print("✏️  Draw one line, then run the next cell.")
m_walk

In [ ]:
#@title ▶️ Run me: fingerprint the walk { display-mode: "form" }
FALLBACK_LINE = "-87.613,41.912  -87.605,41.867  -87.598,41.830"  #@param {type:"string"}
STEPS = 12  #@param {type:"integer"}

def line_points(coords, n):
    # resample a polyline to n evenly spaced points (flat-earth locally, fine for walks)
    Pts  = np.asarray(coords, float)
    lat0 = Pts[:, 1].mean()
    xy   = np.column_stack([Pts[:, 0] * 111.32 * np.cos(np.radians(lat0)),
                            Pts[:, 1] * 110.57])
    seg  = np.hypot(*np.diff(xy, axis=0).T)
    d    = np.r_[0, np.cumsum(seg)]
    t    = np.linspace(0, d[-1], n)
    return (np.column_stack([np.interp(t, d, Pts[:, 0]), np.interp(t, d, Pts[:, 1])]),
            float(d[-1]))

drawn = drawn_geometry(m_walk, "LineString") if "m_walk" in dir() else None
QUERY_LINE = drawn if drawn else [tuple(map(float, p.split(","))) for p in FALLBACK_LINE.split()]
source = "your drawing" if drawn else "fallback line"

pts_q, length_km = line_points(QUERY_LINE, STEPS)
Q = np.full((STEPS, 64), np.nan)
for i, pr in enumerate(sample_image(emb, pts_q, 10)):
    if pr and pr.get("A00") is not None:
        Q[i] = [pr[b] for b in BANDS]

ok = ~np.isnan(Q).any(axis=1)
if (~ok).sum():
    print(f"({(~ok).sum()} of {STEPS} steps had no fingerprint — probably water — dropped)")
assert ok.sum() >= 6, "Too much of this walk is over open water. Draw one mostly over land."

KEPT   = np.where(ok)[0]
Q      = Q[ok]
qmean  = Q.mean(axis=0); qmean /= np.linalg.norm(qmean)

print(f"👣 Walk loaded from {source}: {length_km:.1f} km, {ok.sum()} fingerprinted steps.")
print(f"   Step-to-step similarity along your walk: "
      f"{np.mean(np.sum(Q[:-1] * Q[1:], axis=1)):+.3f} on average")
print("   (near 1.0 = a uniform walk, lower = a walk through changing worlds - harder, better)")

In [ ]:
#@title ▶️ Run me: hunt the planet for your walk (5-15 minutes) { display-mode: "form" }
N_ROUTES   = 5    #@param {type:"integer"}
N_CAND     = 12   #@param {type:"integer"}
ROTATIONS  = 8    #@param {type:"integer"}
EXCLUDE_WALK_KM = 250  #@param {type:"integer"}

def km_offsets(pts):
    lat0, lon0 = pts[:, 1].mean(), pts[:, 0].mean()
    return np.column_stack([(pts[:, 0] - lon0) * 111.32 * np.cos(np.radians(lat0)),
                            (pts[:, 1] - lat0) * 110.57])

def place(offsets, clat, clon, theta_deg):
    th   = np.radians(theta_deg)
    rot  = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    xy   = offsets @ rot.T
    clat = float(np.clip(clat, -78, 78))
    return np.column_stack([clon + xy[:, 0] / (111.32 * np.cos(np.radians(clat))),
                            clat + xy[:, 1] / 110.57])

def km_between(a, b):
    dx = (a["lon"] - b["lon"]) * 111.32 * np.cos(np.radians((a["lat"] + b["lat"]) / 2))
    return float(np.hypot(dx, (a["lat"] - b["lat"]) * 110.57))

def dedupe(rows, min_km, keep):
    out = []
    for r in rows:
        if all(km_between(r, o) > min_km for o in out):
            out.append(r)
        if len(out) == keep:
            break
    return out

def score_batch(placements, scale):
    # one similarity band per step of the walk; each sampled point reads its own step's band
    pts   = np.concatenate([p["pts"] for p in placements])
    props = sample_image(SIMSTACK, pts, scale)
    ns    = len(KEPT)
    for i, p in enumerate(placements):
        vals = [props[i * ns + k][f"s{k:02d}"] for k in range(ns)
                if props[i * ns + k] and props[i * ns + k].get(f"s{k:02d}") is not None]
        p["valid"] = len(vals) / ns
        p["score"] = float(np.mean(vals)) if p["valid"] >= 0.7 else -9.0
    return placements

SIMSTACK = ee.Image.cat([sim_image(emb, Q[k]).rename(f"s{k:02d}") for k in range(len(KEPT))])
offs     = km_offsets(pts_q[ok])
centroid = {"lat": float(pts_q[:, 1].mean()), "lon": float(pts_q[:, 0].mean())}

# ---- stage 1: blur - where is the AVERAGE of this walk common? -------------
print("🌀 Stage 1: scanning the planet at 25 km …")
WORLD = ee.Geometry.Rectangle([-179.9, -75, 179.9, 79], None, False)
samp  = (sim_image(emb, qmean).addBands(ee.Image.pixelLonLat())
         .sample(region=WORLD, scale=25000, numPixels=6000, seed=7)
         .getInfo()["features"])
rows = sorted([{"lat": f["properties"]["latitude"], "lon": f["properties"]["longitude"],
                "score": f["properties"]["sim"]} for f in samp],
              key=lambda r: r["score"], reverse=True)
rows  = [r for r in rows if abs(r["lat"]) < 76 and km_between(r, centroid) > EXCLUDE_WALK_KM]
cands = dedupe(rows, 400, N_CAND)
print(f"   {len(cands)} candidate regions shortlisted "
      f"(best mean-fingerprint similarity {cands[0]['score']:+.3f})")

# ---- stage 2: try the walk everywhere plausible, at 240 m ------------------
GRID = [-30, -15, 0, 15, 30]   # km around each candidate centre
scored = []
for ci, c in enumerate(cands):
    batch = []
    for dy in GRID:
        for dx in GRID:
            for r in range(ROTATIONS):
                clat = c["lat"] + dy / 110.57
                clon = c["lon"] + dx / (111.32 * np.cos(np.radians(c["lat"])))
                theta = 360.0 * r / ROTATIONS
                batch.append({"lat": clat, "lon": clon, "rot": theta,
                              "pts": place(offs, clat, clon, theta)})
    scored += score_batch(batch, 240)
    best = max(p["score"] for p in scored)
    print(f"   ✓ candidate {ci + 1:2d}/{len(cands)}   best walk so far {best:+.3f}")

# ---- stage 3: focus - jitter the survivors at 60 m -------------------------
top = dedupe(sorted([p for p in scored if p["score"] > -9],
                    key=lambda p: p["score"], reverse=True), 100, max(N_ROUTES, 4))
print("🔬 Stage 3: refining the survivors at 60 m …")
refined = []
for t in top:
    batch = []
    for dy in [-6, -3, 0, 3, 6]:
        for dx in [-6, -3, 0, 3, 6]:
            for dth in [-20, -10, 0, 10, 20]:
                clat = t["lat"] + dy / 110.57
                clon = t["lon"] + dx / (111.32 * np.cos(np.radians(t["lat"])))
                theta = t["rot"] + dth
                batch.append({"lat": clat, "lon": clon, "rot": theta,
                              "pts": place(offs, clat, clon, theta)})
    refined += score_batch(batch, 60)

# ---- final: score the finalists at the true 10 m ---------------------------
winners = dedupe(sorted([p for p in refined if p["score"] > -9],
                        key=lambda p: p["score"], reverse=True), 100, N_ROUTES)
final_props = sample_image(SIMSTACK, np.concatenate([w["pts"] for w in winners]), 10)
ns = len(KEPT)
for i, w in enumerate(winners):
    vals = [final_props[i * ns + k].get(f"s{k:02d}") if final_props[i * ns + k] else None
            for k in range(ns)]
    w["stepwise"] = [v if v is not None else np.nan for v in vals]
    w["score10"]  = float(np.nanmean(w["stepwise"]))

winners.sort(key=lambda w: w["score10"], reverse=True)

print("\n" + "=" * 74)
print(f"  THE TOP {len(winners)} GHOSTS OF YOUR {length_km:.1f} KM WALK")
print("=" * 74)
for i, w in enumerate(winners):
    print(f"  #{i+1}  sim {w['score10']:+.3f}   heading {w['rot']:5.1f}°   "
          f"({w['lat']:.4f}, {w['lon']:.4f})   {maps_link(w['lat'], w['lon'])}")
print("=" * 74)

In [ ]:
#@title ▶️ Run me: the ghosts, on a map { display-mode: "form" }
m_ghost = new_map([20, 0], 2)
m_ghost.add_basemap("CartoDB.DarkMatter")
m_ghost.addLayer(ee.Geometry.LineString([[float(x), float(y)] for x, y in pts_q]),
                 {"color": "white"}, "your walk")
GHOST_COLORS = ["C83232", "E8A33D", "3d8f6f", "7a4fa3", "3d5a80", "b8862b", "c96f9a"]
for i, w in enumerate(winners):
    m_ghost.addLayer(ee.Geometry.LineString([[float(x), float(y)] for x, y in w["pts"]]),
                     {"color": GHOST_COLORS[i % len(GHOST_COLORS)]},
                     f"ghost #{i+1}  sim {w['score10']:+.3f}")

# how well does each ghost track your walk, step by step?
fig, ax = plt.subplots(figsize=(9.2, 4.2))
for i, w in enumerate(winners):
    ax.plot(range(1, len(KEPT) + 1), w["stepwise"], marker="o", ms=5, lw=2,
            color="#" + GHOST_COLORS[i % len(GHOST_COLORS)], label=f"ghost #{i+1}")
ax.axhline(1.0, color="#8b8981", lw=1, ls=":")
ax.set_xlabel("step along the walk"); ax.set_ylabel("similarity to your step")
ax.set_title("Do the ghosts keep up, or fake it on average?")
ax.legend(fontsize=9.5); ax.grid(alpha=0.5)
caption(ax, "A flat high line is a true ghost. A spiky line matched your average but "
            "not your sequence - the shore where you had a park.")
plt.show()
m_ghost

### 👀 What to look for

- **Sanity check:** set `EXCLUDE_WALK_KM` to 0 and rerun — your own walk should come back
  as ghost #1 with similarity ≈ 1.0. If it does not, the search is broken; file a bug
  against yourself.
- **Flat vs spiky** in the step chart is the whole story. Matching a walk's *average* is
  easy; matching its *sequence* — the order in which worlds change — is the hard part and
  the honest test.
- Walks that cross contrasting worlds (water → park → towers) produce far more specific
  ghosts than uniform ones. A walk through endless suburb has ten thousand ghosts.

---

## 🔭 Research notes, and the road to the toy app

**Known limits of v0 (be suspicious of all results accordingly):**

- The zoom ladder inherits the pyramid's blur: an isolated 10 m twin inside a different
  neighbourhood can be missed. Continents are hand-drawn; border zones leak a little.
- Ghost walks are rigid: same length, same shape, rotation only. No mirroring, no
  stretching, and the candidate stage (mean fingerprint at 25 km) can miss a region whose
  *sequence* matches but whose average does not.
- Everything is one year of one embedding model. Twins can drift year to year.

**Research directions worth a notebook each:**

- **Elastic ghosts** — dynamic time warping over the similarity field, so a 5 km walk can
  match a 7 km walk with the same story.
- **Trajectory twins** — match places by their year-over-year *motion* through embedding
  space, not their position: find every place currently becoming what your city became.
- **Anti-portals** — the least similar place on Earth. Surprisingly good at teaching what
  the embedding actually encodes.
- **Similarity isochrones** — how far do you have to travel from home before similarity
  drops below 0.5? Maps of "how far away different begins".

**The toy app sketch** (this notebook is the algorithm spec):

1. **Offline:** export the global embedding at ~1–5 km once (a few hundred MB), build an
   ANN index (FAISS / ScaNN). Global candidate search becomes a ~10 ms lookup instead of a
   25 km Earth Engine scan.
2. **Online:** the index serves instant coarse answers; Earth Engine (or pre-exported
   tiles) refines the winners to 10 m exactly like `best_match` does here.
3. **Frontend:** a map with three tools — drop a portal, draw a region, draw a walk —
   backed by the three searches above. Ghost-walk jobs run async with a progress bar;
   popular portals get cached.
4. **The share card:** two satellite photos side by side — "you are here / you could be
   here" — with the similarity score. That is the whole product.